# DBRepo Road Traffic Accident ETL Pipeline


In [ ]:
import os
import time
import logging
import pandas as pd
import numpy as np
import requests
from typing import Dict, List, Any, Optional, Tuple
from getpass import getpass
from requests.auth import HTTPBasicAuth

# --- Configuration Setup ---
logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')
logger = logging.getLogger(__name__)

DATA_DIR = "../data"

CONFIG = {
    "BASE_URL": "https://test.dbrepo.tuwien.ac.at/api/v1",
    "DATABASE_ID": "b23492bd-f66d-4663-a1f5-f296767dbbdc",
    "CLEAR_FACT_TABLES_BEFORE_LOAD": True,  
    "CLEAR_LOOKUP_TABLES_BEFORE_LOAD": False,
    
    # Mapped to the data folder
    "GUIDANCE_FILE": f"{DATA_DIR}/Guidance.csv",
    "TEXT_FILES": [f"{DATA_DIR}/{year}.csv" for year in range(2009, 2017)],
    "CODE_FILES": [f"{DATA_DIR}/{year}.csv" for year in range(2017, 2020)],
    
    "PAGE_SIZE": 1000,
    "SKIP_DUPLICATE_ROWS": True
}

DB_LOOKUP_MAPPING = {
    "1st Road Class":       ("road_class", "road_class_id"),
    "Road Surface":         ("road_surface", "road_surface_id"),
    "Lighting Conditions":  ("lighting", "lighting_id"),
    "Weather Conditions":   ("weather", "weather_id"),
    "Casualty Class":       ("casualty_class", "casualty_class_id"),
    "Casualty Severity":    ("casualty_severity", "casualty_severity_id"),
    "Sex of Casualty":      ("sex", "sex_id"),
    "Type of Vehicle":      ("vehicle", "vehicle_id"),
}

In [ ]:
class DBRepoClient:
    """Handles API interactions, paginated reads, and resilient writes to DBRepo."""

    def __init__(self, base_url: str, database_id: str, username: str, password: str):
        self.base_url = base_url.rstrip("/")
        self.database_id = database_id
        
        self.session = requests.Session()
        self.session.auth = HTTPBasicAuth(username, password)
        self.table_map = self._refresh_table_map()

    def _refresh_table_map(self) -> Dict[str, str]:
        url = f"{self.base_url}/database/{self.database_id}"
        response = self.session.get(url, timeout=10)
        response.raise_for_status()
        
        mapping = {t["name"]: t["id"] for t in response.json().get("tables", [])}
        logger.info(f"Table Mapping retrieved: {list(mapping.keys())}")
        return mapping

    @staticmethod
    def _clean_record(record: dict) -> dict:
        cleaned = {}
        for key, value in record.items():
            if pd.isna(value):
                cleaned[key] = None
            elif hasattr(value, "item"):
                cleaned[key] = value.item()
            else:
                cleaned[key] = value
        return cleaned

    @staticmethod
    def _is_duplicate_error(response: Any) -> bool:
        text = getattr(response, "text", "").lower()
        return "duplicate entry" in text or "for key 'primary'" in text or 'for key "primary"' in text

    def get_table_rows(self, table_name: str, size: int = 1000) -> List[Dict]:
        table_id = self.table_map.get(table_name)
        if not table_id:
            return []

        rows, page = [], 0
        while True:
            url = f"{self.base_url}/database/{self.database_id}/table/{table_id}/data?page={page}&size={size}"
            try:
                res = self.session.get(url, headers={"Accept": "application/json"}, timeout=5)
            except requests.exceptions.RequestException:
                break

            if res.status_code not in [200, 201, 202]:
                break

            payload = res.json()
            page_rows = payload if isinstance(payload, list) else payload.get("content", [])
            if not page_rows:
                break

            rows.extend(page_rows)
            if len(page_rows) < size:
                break
            page += 1

        return rows

    def post_one_row(self, table_name: str, record: dict, retries: int = 3) -> Tuple[bool, Any]:
        table_id = self.table_map.get(table_name)
        url = f"{self.base_url}/database/{self.database_id}/table/{table_id}/data"
        body = {"data": self._clean_record(record)}
        
        last_res = None
        for attempt in range(retries):
            try:
                res = self.session.post(url, json=body, headers={"Content-Type": "application/json"}, timeout=5)
                last_res = res
                if res.status_code in [200, 201, 202]:
                    return True, res
                if res.status_code in [429, 500, 502, 503, 504]:
                    time.sleep(0.5 * (2 ** attempt))
                    continue
                break
            except requests.exceptions.RequestException as e:
                last_res = type('obj', (object,), {'text': str(e), 'status_code': 500})
                time.sleep(0.5 * (2 ** attempt))
                
        return False, last_res

    def delete_one_row(self, table_name: str, pk_record: dict) -> bool:
        table_id = self.table_map.get(table_name)
        url = f"{self.base_url}/database/{self.database_id}/table/{table_id}/data"
        payload = {"data": self._clean_record(pk_record)}
        
        res = self.session.delete(url, json=payload, headers={"Content-Type": "application/json"}, timeout=10)
        return res.status_code in [200, 201, 202, 204, 404]

    def clear_table(self, table_name: str, pk_cols: List[str]) -> Tuple[int, int]:
        logger.info(f"Clearing table '{table_name}'...")
        rows = self.get_table_rows(table_name, size=CONFIG["PAGE_SIZE"])
        deleted, failed = 0, 0
        for row in rows:
            pk_record = {col: row.get(col) for col in pk_cols}
            if any(v is None for v in pk_record.values()):
                continue
            if self.delete_one_row(table_name, pk_record):
                deleted += 1
            else:
                failed += 1
        logger.info(f"Cleared '{table_name}': {deleted} deleted, {failed} failed")
        return deleted, failed

    def upload_dataframe(self, table_name: str, df: pd.DataFrame, skip_duplicate_errors: bool = True) -> Tuple[int, int, int]:
        records = df.where(pd.notnull(df), None).to_dict(orient="records")
        logger.info(f"Uploading {len(records)} rows to '{table_name}'...")

        success, failed, skipped = 0, 0, 0
        for i, record in enumerate(records):
            ok, res = self.post_one_row(table_name, record)
            if ok:
                success += 1
            elif skip_duplicate_errors and self._is_duplicate_error(res):
                skipped += 1
            else:
                failed += 1
                logger.error(f"Failed row in '{table_name}': {getattr(res, 'text', '')}")
                break

            if (success + skipped + failed) % 1000 == 0:
                logger.info(f" Processed {success + skipped + failed} rows...")

        return success, failed, skipped

In [ ]:
def parse_guidance_file(filepath: str) -> Tuple[Dict, Dict]:
    if not os.path.exists(filepath):
        logger.warning(f"File not found: {filepath}")
        return {}, {}
        
    lookups = {}
    with open(filepath, "r", encoding="cp1252") as f:
        current_lookup = None
        for line in f:
            line = line.strip()
            if not line or line == ",":
                continue

            parts = [p.strip() for p in line.split(",")]
            if parts[0] and not parts[0].isdigit() and parts[0] != "[Age given in years]" and "Desc" in line:
                current_lookup = parts[0].strip()
                lookups[current_lookup] = {}
            elif current_lookup and parts[0].isdigit() and len(parts) > 1:
                lookups[current_lookup][int(parts[0])] = parts[1].strip()

    reverse_lookups = {
        guide_name: {str(v).lower().strip(): k for k, v in lookup_dict.items()} 
        for guide_name, lookup_dict in lookups.items()
    }
    return lookups, reverse_lookups

def load_and_consolidate_datasets(text_files: List[str], code_files: List[str], reverse_lookups: dict) -> pd.DataFrame:
    df_list = []
    def safe_map(val: Any, lookup_dict: Dict[str, int]) -> float:
        return lookup_dict.get(str(val).strip().lower(), np.nan) if pd.notna(val) else np.nan

    for file in text_files:
        if not os.path.exists(file):
            logger.warning(f"File not found: {file}")
            continue
        df = pd.read_csv(file, skipinitialspace=True, low_memory=False, encoding='cp1252')
        
        col_mapping = {
            'Reference Number': 'reference_number', 'Easting': 'easting', 'Grid Ref: Easting': 'easting',
            'Northing': 'northing', 'Grid Ref: Northing': 'northing', 'Number of Vehicles': 'number_of_vehicles',
            'Accident Date': 'date', 'Time (24hr)': 'time', 'Time': 'time', '1st Road Class': 'road_class',
            'Road Surface': 'road_surface', 'Lighting Conditions': 'lighting', 'Weather Conditions': 'weather',
            'Casualty Class': 'casualty_class', 'Casualty Severity': 'casualty_severity', 'Sex of Casualty': 'sex',
            'Age of Casualty': 'age', 'Type of Vehicle': 'vehicle',
            'Unnamed: 11': 'casualty_class', 'Unnamed: 15': 'vehicle'
        }
        df = df.rename(columns=col_mapping)
        
        df['road_class_id'] = df['road_class'].apply(lambda x: safe_map(x, reverse_lookups.get('1st Road Class', {})))
        df['road_surface_id'] = df['road_surface'].apply(lambda x: safe_map(x, reverse_lookups.get('Road Surface', {})))
        df['lighting_id'] = df['lighting'].apply(lambda x: safe_map(x, reverse_lookups.get('Lighting Conditions', {})))
        df['weather_id'] = df['weather'].apply(lambda x: safe_map(x, reverse_lookups.get('Weather Conditions', {})))
        df['casualty_class_id'] = df['casualty_class'].apply(lambda x: safe_map(x, reverse_lookups.get('Casualty Class', {})))
        df['casualty_severity_id'] = df['casualty_severity'].apply(lambda x: safe_map(x, reverse_lookups.get('Casualty Severity', {})))
        df['sex_id'] = df['sex'].apply(lambda x: safe_map(x, reverse_lookups.get('Sex of Casualty', {})))
        df['vehicle_id'] = df['vehicle'].apply(lambda x: safe_map(x, reverse_lookups.get('Type of Vehicle', {})))
        df_list.append(df)

    for file in code_files:
        if not os.path.exists(file):
            logger.warning(f"File not found: {file}")
            continue
        df = pd.read_csv(file, skipinitialspace=True, low_memory=False, encoding='cp1252')
        col_mapping = {
            'Reference Number': 'reference_number', 'Number of vehicles': 'number_of_vehicles',
            'Date': 'date', 'Time': 'time', 'Road class': 'road_class_id', 'Road surface description': 'road_surface_id',
            'Lighting conditions': 'lighting_id', 'Weather conditions': 'weather_id', 'Casualty Class': 'casualty_class_id',
            'Sex of Casualty': 'sex_id', 'Age of Casualty': 'age', 'Casualty Severity': 'casualty_severity_id',
            'Type of vehicle 1': 'vehicle_id'
        }
        df = df.rename(columns=col_mapping)
        if 'easting' not in df.columns: df['easting'] = 0
        if 'northing' not in df.columns: df['northing'] = 0
        df_list.append(df)

    return pd.concat(df_list, ignore_index=True) if df_list else pd.DataFrame()

def split_and_prepare_fact_tables(df: pd.DataFrame) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    if df.empty:
        raise ValueError("Dataset is empty. Cannot process.")

    df['date'] = pd.to_datetime(df['date'], format='mixed', dayfirst=True, errors='coerce').dt.strftime('%Y-%m-%d')
    
    def normalize_time(s):
        s = str(s).replace(".0", "").strip()
        if ":" in s: return int(s.split(":")[0]) * 100 + int(s.split(":")[1])
        return int(s) if s.isdigit() else 0
        
    df['time'] = df.get('time', 0).apply(normalize_time).astype(int)

    for col in ['reference_number', 'easting', 'northing', 'number_of_vehicles', 'age']:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0).astype(int)

    id_cols = ['road_class_id', 'road_surface_id', 'lighting_id', 'weather_id',
               'casualty_class_id', 'casualty_severity_id', 'sex_id', 'vehicle_id']
    for col in id_cols:
        df[col] = pd.to_numeric(df.get(col, np.nan), errors='coerce').fillna(1).astype(int)

    df['road_surface_id'] = df['road_surface_id'].replace(0, 9)
    df['person_id'] = range(1, len(df) + 1)

    accident_df = df[[
        'reference_number', 'easting', 'northing', 'number_of_vehicles', 
        'date', 'time', 'road_class_id', 'road_surface_id', 'lighting_id', 'weather_id'
    ]].drop_duplicates('reference_number').copy()

    person_df = df[['person_id', 'sex_id', 'age']].copy()

    accident_casualty_df = df[[
        'reference_number', 'person_id', 'casualty_class_id', 'casualty_severity_id', 'vehicle_id'
    ]].copy()

    return accident_df, person_df, accident_casualty_df

In [ ]:
# ============================================================
# Execution Runner
# ============================================================

print("--- DBRepo Road Traffic Accident ETL ---")
username = input("DBRepo username: ").strip()
password = getpass("DBRepo password: ")

client = DBRepoClient(CONFIG["BASE_URL"], CONFIG["DATABASE_ID"], username, password)

# 1. Clear Old Data
if CONFIG["CLEAR_FACT_TABLES_BEFORE_LOAD"]:
    client.clear_table("accident_casualty", ["reference_number", "person_id"])
    client.clear_table("person", ["person_id"])
    client.clear_table("accident", ["reference_number"])

if CONFIG["CLEAR_LOOKUP_TABLES_BEFORE_LOAD"]:
    for tbl, pk in [("vehicle", "vehicle_id"), ("sex", "sex_id"), ("casualty_severity", "casualty_severity_id")]:
        client.clear_table(tbl, [pk])

# 2. Populate Lookups
lookups, reverse_lookups = parse_guidance_file(CONFIG["GUIDANCE_FILE"])
for guide_name, (table_name, pk_col) in DB_LOOKUP_MAPPING.items():
    if guide_name not in lookups:
        continue
    existing_ids = {int(r[pk_col]) for r in client.get_table_rows(table_name) if r.get(pk_col) is not None}
    for key, desc in lookups[guide_name].items():
        if key not in existing_ids:
            client.post_one_row(table_name, {pk_col: key, "description": str(desc)})

# 3. Transform Fact Data
raw_df = load_and_consolidate_datasets(CONFIG["TEXT_FILES"], CONFIG["CODE_FILES"], reverse_lookups)
acc_df, per_df, cas_df = split_and_prepare_fact_tables(raw_df)

# 4. Upload Tables (Safe FK Order)
logger.info("Uploading 'accident' table...")
if client.upload_dataframe('accident', acc_df, skip_duplicate_errors=CONFIG["SKIP_DUPLICATE_ROWS"])[1] > 0:
    logger.error("Accident upload failed. Halting pipeline.")
else:
    logger.info("Uploading 'person' table...")
    if client.upload_dataframe('person', per_df, skip_duplicate_errors=CONFIG["SKIP_DUPLICATE_ROWS"])[1] > 0:
        logger.error("Person upload failed. Halting pipeline.")
    else:
        logger.info("Uploading 'accident_casualty' table...")
        client.upload_dataframe('accident_casualty', cas_df, skip_duplicate_errors=CONFIG["SKIP_DUPLICATE_ROWS"])
        logger.info("===== PIPELINE COMPLETE =====")